In [ ]:
!pip install xarray netCDF4 huggingface_hub matplotlib

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("paulhoehn/Sen12Landslides", repo_type="dataset")

s2_files = [f for f in files if f.startswith("data_harmonized/s2/")]
for f in s2_files:
    print(f)

data_harmonized/s2/s2_part01.tar.gz
data_harmonized/s2/s2_part02.tar.gz
data_harmonized/s2/s2_part03.tar.gz
data_harmonized/s2/s2_part04.tar.gz
data_harmonized/s2/s2_part05.tar.gz
data_harmonized/s2/s2_part06.tar.gz
data_harmonized/s2/s2_part07.tar.gz
data_harmonized/s2/s2_part08.tar.gz
data_harmonized/s2/s2_part09.tar.gz
data_harmonized/s2/s2_part10.tar.gz
data_harmonized/s2/s2_part11.tar.gz
data_harmonized/s2/s2_part12.tar.gz
data_harmonized/s2/s2_part13.tar.gz
data_harmonized/s2/s2_part14.tar.gz
data_harmonized/s2/s2_part15.tar.gz
data_harmonized/s2/s2_part16.tar.gz
data_harmonized/s2/s2_part17.tar.gz
data_harmonized/s2/s2_part18.tar.gz
data_harmonized/s2/s2_part19.tar.gz
data_harmonized/s2/s2_part20.tar.gz
data_harmonized/s2/s2_part21.tar.gz
data_harmonized/s2/s2_part22.tar.gz
data_harmonized/s2/s2_part23.tar.gz
data_harmonized/s2/s2_part24.tar.gz
data_harmonized/s2/s2_part25.tar.gz
data_harmonized/s2/s2_part26.tar.gz
data_harmonized/s2/s2_part27.tar.gz
data_harmonized/s2/s2_part28

In [ ]:
from huggingface_hub import hf_hub_url
import requests

sample_file = "data_harmonized/s2/s2_part01.tar.gz"
url = hf_hub_url(repo_id="paulhoehn/Sen12Landslides", filename=sample_file, repo_type="dataset")
response = requests.head(url, allow_redirects=True)
size_mb = int(response.headers.get('content-length', 0)) / (1024 * 1024)
print(f"{sample_file}: {size_mb:.2f} MB")

data_harmonized/s2/s2_part01.tar.gz: 1405.94 MB


In [ ]:
from huggingface_hub import hf_hub_download

downloaded_path = hf_hub_download(
    repo_id="paulhoehn/Sen12Landslides",
    filename="data_harmonized/s2/s2_part01.tar.gz",
    repo_type="dataset",
    local_dir="/content/data"
)

print("Downloaded to:", downloaded_path)

Downloaded to: /content/data/data_harmonized/s2/s2_part01.tar.gz


In [ ]:
import tarfile

extract_path = "/content/data/data_harmonized/s2"

with tarfile.open(downloaded_path, "r:gz") as tar:
    tar.extractall(path=extract_path)

print("Extraction complete.")

/tmp/ipykernel_17755/4091362273.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete.


In [ ]:
import os

nc_files = [f for f in os.listdir(extract_path) if f.endswith(".nc")]
print(f"Number of .nc files extracted: {len(nc_files)}")
print("First 5 filenames:")
for f in nc_files[:5]:
    print(" -", f)

Number of .nc files extracted: 1000
First 5 filenames:
 - chimanimani_s2_1152.nc
 - chimanimani_s2_1820.nc
 - chimanimani_s2_1118.nc
 - chimanimani_s2_683.nc
 - chimanimani_s2_1878.nc


In [ ]:
import xarray as xr

sample_path = os.path.join(extract_path, nc_files[0])
ds = xr.open_dataset(sample_path)
print(ds)

<xarray.Dataset> Size: 6MB
Dimensions:      (time: 15, x: 128, y: 128)
Coordinates:
  * time         (time) datetime64[ns] 120B 2018-09-11 2018-10-01 ... 2019-09-06
  * x            (x) float64 1kB 4.826e+05 4.826e+05 ... 4.838e+05 4.838e+05
  * y            (y) float64 1kB 7.835e+06 7.835e+06 ... 7.834e+06 7.834e+06
Data variables: (12/14)
    B02          (time, x, y) int16 492kB ...
    B03          (time, x, y) int16 492kB ...
    B04          (time, x, y) int16 492kB ...
    B05          (time, x, y) int16 492kB ...
    B06          (time, x, y) int16 492kB ...
    B07          (time, x, y) int16 492kB ...
    ...           ...
    B11          (time, x, y) int16 492kB ...
    B12          (time, x, y) int16 492kB ...
    SCL          (time, x, y) int16 492kB ...
    MASK         (time, x, y) uint8 246kB ...
    DEM          (time, x, y) int16 492kB ...
    spatial_ref  int64 8B ...
Attributes:
    ann_id:           170,171,172,174,176,177,178,3008
    ann_bbox:         (482560.0,

In [ ]:
# check a skipped file's attributes directly
ds_check = xr.open_dataset(os.path.join(extract_path, "chimanimani_s2_1621.nc"))
print(ds_check.attrs)
ds_check.close()

{'ann_id': '', 'ann_bbox': '', 'event_date': '', 'date_confidence': '', 'pre_post_dates': '', 'annotated': 'False', 'satellite': 's2', 'center_lat': np.float64(7840640.0), 'center_lon': np.float64(493440.0), 'crs': 'EPSG:32736'}


In [ ]:
import numpy as np
import xarray as xr
import os
import ast

def get_pre_post(ds):
    raw = ds.attrs.get('pre_post_dates', None)
    if not raw:
        return None, None
    try:
        d = ast.literal_eval(raw)
        return d.get('pre'), d.get('post')
    except Exception:
        return None, None

nc_dir = extract_path # Define nc_dir
X_data, y_labels = [], []

for fname in nc_files:
    try:
        ds = xr.open_dataset(os.path.join(nc_dir, fname))
        pre_idx, post_idx = get_pre_post(ds)
        t_idx = post_idx if post_idx is not None else ds.sizes['time'] - 1

        r = ds['B04'].isel(time=t_idx).values
        g = ds['B03'].isel(time=t_idx).values
        b = ds['B02'].isel(time=t_idx).values
        rgb = np.clip(np.stack([r, g, b], axis=-1).astype(float) / 3000, 0, 1)

        mask = ds['MASK'].isel(time=t_idx).values
        label = 1 if mask.sum() > 0 else 0

        X_data.append(rgb)
        y_labels.append(label)
        ds.close()
    except Exception:
        continue

X_data = np.array(X_data)
y_labels = np.array(y_labels)
print("Shape:", X_data.shape)
print("Label distribution:", np.bincount(y_labels))

Shape: (1000, 128, 128, 3)
Label distribution: [609 391]


In [ ]:
import tensorflow as tf
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
